In [ ]:
import re
import time
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from pypdf import PdfReader
from tqdm import tqdm


# =========================
# 1. 你需要改的地方
# =========================

INPUT_FILE = Path("/Users/tanglikun/Desktop/all_a_share_stock_list_20260705_130506.csv")

OUTPUT_DIR = Path("/Users/tanglikun/Desktop/AI_annual_report_screen_2025")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_EXCEL = OUTPUT_DIR / "A股AI产业链年报初筛结果_2025.xlsx"
OUTPUT_CSV = OUTPUT_DIR / "A股AI产业链年报初筛结果_2025.csv"

PDF_DIR = OUTPUT_DIR / "annual_report_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

TEXT_DIR = OUTPUT_DIR / "annual_report_texts"
TEXT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = "2025"

# 如果你想先测试前50家公司，把 None 改成 50
MAX_COMPANIES = 50

# 每家公司之间暂停，避免访问太快
SLEEP_SECONDS = 0.8


# =========================
# 2. AI产业链关键词
# =========================

AI_KEYWORDS = {
    "AI基础层": [
        "人工智能", "AI", "算力", "智能算力", "智算", "大模型算力",
        "GPU", "GPGPU", "ASIC", "FPGA", "AI芯片", "推理芯片", "训练芯片",
        "服务器", "AI服务器", "智算中心", "数据中心", "IDC", "云计算",
        "边缘计算", "液冷", "液冷服务器", "冷板液冷", "浸没式液冷",
        "光模块", "CPO", "硅光", "光通信", "交换机", "路由器",
        "PCB", "高速PCB", "HDI", "封装基板", "先进封装", "HBM",
        "存储芯片", "DRAM", "NAND", "连接器", "电源模块", "散热"
    ],
    "模型与软件层": [
        "大模型", "基础模型", "行业大模型", "垂直大模型", "生成式AI", "AIGC",
        "机器学习", "深度学习", "神经网络", "自然语言处理", "NLP",
        "计算机视觉", "机器视觉", "语音识别", "语音合成", "多模态",
        "知识图谱", "智能算法", "算法模型", "模型训练", "模型推理",
        "数据标注", "数据治理", "数据要素", "训练数据", "AI平台",
        "MaaS", "智能中台", "智能决策", "智能客服", "OCR", "RPA"
    ],
    "AI应用层": [
        "智能驾驶", "自动驾驶", "辅助驾驶", "ADAS", "智能座舱",
        "车路协同", "智能网联", "机器人", "工业机器人", "人形机器人",
        "服务机器人", "协作机器人", "智能制造", "工业互联网",
        "缺陷检测", "预测性维护", "智慧医疗", "AI医疗", "医学影像",
        "智能诊断", "AI制药", "智慧金融", "智能投顾", "智慧城市",
        "智能安防", "智能家居", "智慧教育", "智慧政务", "智慧交通",
        "智慧能源", "智慧矿山", "智慧物流", "智能仓储", "数字人", "虚拟人"
    ],
    "AI配套生态": [
        "传感器", "激光雷达", "毫米波雷达", "摄像头模组", "图像传感器",
        "智能终端", "AR", "VR", "MR", "智能穿戴", "无人机",
        "系统集成", "AI解决方案", "智能化解决方案", "数字化解决方案"
    ],
}

# 一些太宽的词，单独处理，避免“智能水表”这类误伤太多
WEAK_KEYWORDS = [
    "智能", "数字化", "自动化", "信息化", "云平台", "数据平台"
]

# 明显非AI行业。如果这些行业没有命中AI关键词，优先判为“否”
OBVIOUS_NON_AI_INDUSTRY_WORDS = [
    "白酒", "啤酒", "葡萄酒", "黄酒", "饮料", "乳品", "食品",
    "调味品", "休闲食品", "肉制品", "养殖", "种植", "饲料",
    "农业", "林业", "渔业", "旅游", "酒店", "餐饮", "景区",
    "房地产", "物业", "百货", "零售", "服装", "家纺", "珠宝",
    "化妆品", "造纸", "包装印刷", "水泥", "煤炭", "钢铁",
    "化肥", "农药", "港口", "航运", "公路", "铁路运输"
]


# =========================
# 3. 工具函数
# =========================

def normalize_code(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.replace("bj", "").replace("sh", "").replace("sz", "")
    s = re.sub(r"\D", "", s)
    if len(s) > 6:
        s = s[-6:]
    return s.zfill(6) if s else ""


def guess_exchange_column(code):
    if code.startswith(("600", "601", "603", "605", "688", "689")):
        return "sse"
    if code.startswith(("000", "001", "002", "003", "300", "301")):
        return "szse"
    if code.startswith(("430", "831", "832", "833", "834", "835", "836", "837", "838", "839", "870", "871", "872", "873", "920")):
        return "bj"
    return ""


def find_column(df, candidates):
    cols = list(df.columns)
    for c in candidates:
        if c in cols:
            return c
    for col in cols:
        for c in candidates:
            if c in str(col):
                return col
    return None


def read_stock_list(path):
    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path, dtype=str)
    else:
        df = pd.read_csv(path, dtype=str, encoding="utf-8-sig")

    code_col = find_column(df, ["股票代码", "证券代码", "代码", "symbol", "code"])
    name_col = find_column(df, ["股票简称", "证券简称", "名称", "简称", "name"])
    industry_col = find_column(df, ["所属行业", "行业", "行业名称", "板块"])

    if code_col is None:
        raise ValueError("没有找到股票代码列。请确认表里有：股票代码/证券代码/代码 这一类列名。")
    if name_col is None:
        raise ValueError("没有找到股票简称列。请确认表里有：股票简称/证券简称/名称 这一类列名。")

    out = pd.DataFrame()
    out["股票代码"] = df[code_col].apply(normalize_code)
    out["股票简称"] = df[name_col].astype(str).str.strip()

    if industry_col:
        out["所属行业"] = df[industry_col].astype(str).str.strip()
    else:
        out["所属行业"] = ""

    out = out[out["股票代码"].str.len() == 6].drop_duplicates("股票代码")
    out = out.reset_index(drop=True)
    return out


def cninfo_query_annual_report(code, name):
    url = "https://www.cninfo.com.cn/new/hisAnnouncement/query"

    columns_to_try = []
    guessed = guess_exchange_column(code)
    if guessed:
        columns_to_try.append(guessed)
    columns_to_try += ["szse", "sse", "bj"]

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.cninfo.com.cn/new/commonUrl/pageOfSearch?url=disclosure/list/search",
        "Accept": "application/json, text/javascript, */*; q=0.01",
    }

    for column in columns_to_try:
        data = {
            "stock": code,
            "searchkey": "",
            "plate": "",
            "category": "category_ndbg_szsh",
            "trade": "",
            "column": column,
            "pageNum": 1,
            "pageSize": 30,
            "tabName": "fulltext",
            "sortName": "",
            "sortType": "",
            "limit": "",
            "seDate": f"{YEAR}-01-01~{int(YEAR)+1}-12-31",
        }

        try:
            r = requests.post(url, headers=headers, data=data, timeout=20)
            if r.status_code != 200:
                continue

            js = r.json()
            anns = js.get("announcements", []) or []

            candidates = []
            for ann in anns:
                title = ann.get("announcementTitle", "")
                adjunct_url = ann.get("adjunctUrl", "")
                announcement_time = ann.get("announcementTime", "")

                title_clean = re.sub(r"<.*?>", "", title)

                if f"{YEAR}年年度报告" not in title_clean:
                    continue
                if any(x in title_clean for x in ["摘要", "英文", "更正", "取消", "已取消", "修订", "补充"]):
                    continue
                if not adjunct_url.lower().endswith(".pdf"):
                    continue

                candidates.append({
                    "公告标题": title_clean,
                    "年报链接": "https://static.cninfo.com.cn/" + adjunct_url,
                    "公告日期": announcement_time,
                    "交易所栏目": column,
                })

            if candidates:
                return candidates[0]

        except Exception:
            continue

    return None


def download_pdf(url, save_path):
    if save_path.exists() and save_path.stat().st_size > 10000:
        return True

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.cninfo.com.cn/",
    }

    try:
        r = requests.get(url, headers=headers, timeout=40)
        if r.status_code == 200 and len(r.content) > 10000:
            save_path.write_bytes(r.content)
            return True
    except Exception:
        return False

    return False


def extract_pdf_text(pdf_path, text_path):
    if text_path.exists() and text_path.stat().st_size > 1000:
        return text_path.read_text(encoding="utf-8", errors="ignore")

    try:
        reader = PdfReader(str(pdf_path))
        parts = []
        for page in reader.pages:
            try:
                txt = page.extract_text() or ""
                parts.append(txt)
            except Exception:
                continue

        text = "\n".join(parts)
        text_path.write_text(text, encoding="utf-8")
        return text
    except Exception:
        return ""


def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    pieces = re.split(r"[。；;！!？?\n]", text)
    return [p.strip() for p in pieces if len(p.strip()) >= 8]


def find_matches(text):
    matches = []
    sentences = split_sentences(text)

    for layer, kws in AI_KEYWORDS.items():
        for kw in kws:
            if kw in text:
                hit_sentences = []
                for s in sentences:
                    if kw in s:
                        hit_sentences.append(s[:180])
                        if len(hit_sentences) >= 3:
                            break

                matches.append({
                    "关键词": kw,
                    "产业链环节": layer,
                    "命中句子": "；".join(hit_sentences)
                })

    weak_hits = []
    for kw in WEAK_KEYWORDS:
        if kw in text:
            for s in sentences:
                if kw in s:
                    weak_hits.append({
                        "关键词": kw,
                        "产业链环节": "弱相关词",
                        "命中句子": s[:180]
                    })
                    break

    return matches, weak_hits


def is_obvious_non_ai_industry(industry, name):
    combined = f"{industry} {name}"
    return any(w in combined for w in OBVIOUS_NON_AI_INDUSTRY_WORDS)


def classify_company(row, report_info, text):
    industry = str(row.get("所属行业", ""))
    name = str(row.get("股票简称", ""))

    if not report_info:
        return {
            "与AI产业链是否有关": "不确定",
            "AI产业链环节": "",
            "命中关键词": "",
            "命中句子": "",
            "命中章节": "",
            "确定性等级": "低",
            "判断原因": "未找到2025年年度报告，无法判断",
        }

    if not text or len(text) < 1000:
        return {
            "与AI产业链是否有关": "不确定",
            "AI产业链环节": "",
            "命中关键词": "",
            "命中句子": "",
            "命中章节": "",
            "确定性等级": "低",
            "判断原因": "年报PDF文字提取失败或文本过短",
        }

    strong_matches, weak_matches = find_matches(text)

    if strong_matches:
        layers = sorted(set(m["产业链环节"] for m in strong_matches))
        keywords = []
        sentences = []

        for m in strong_matches:
            if m["关键词"] not in keywords:
                keywords.append(m["关键词"])
            if m["命中句子"]:
                sentences.append(m["命中句子"])

        if len(strong_matches) >= 3:
            confidence = "高"
        else:
            confidence = "中"

        return {
            "与AI产业链是否有关": "是",
            "AI产业链环节": "、".join(layers),
            "命中关键词": "、".join(keywords[:30]),
            "命中句子": "；".join(sentences[:5]),
            "命中章节": "年报全文关键词命中",
            "确定性等级": confidence,
            "判断原因": "年报中命中AI产业链强关键词",
        }

    if weak_matches:
        if is_obvious_non_ai_industry(industry, name):
            return {
                "与AI产业链是否有关": "否",
                "AI产业链环节": "",
                "命中关键词": "、".join(sorted(set(m["关键词"] for m in weak_matches))),
                "命中句子": "；".join([m["命中句子"] for m in weak_matches[:3]]),
                "命中章节": "年报全文弱关键词命中",
                "确定性等级": "低",
                "判断原因": "仅命中宽泛词，且行业明显偏非AI",
            }

        return {
            "与AI产业链是否有关": "不确定",
            "AI产业链环节": "弱相关",
            "命中关键词": "、".join(sorted(set(m["关键词"] for m in weak_matches))),
            "命中句子": "；".join([m["命中句子"] for m in weak_matches[:3]]),
            "命中章节": "年报全文弱关键词命中",
            "确定性等级": "低",
            "判断原因": "仅命中智能/数字化等宽泛词，需要人工复核",
        }

    if is_obvious_non_ai_industry(industry, name):
        return {
            "与AI产业链是否有关": "否",
            "AI产业链环节": "",
            "命中关键词": "",
            "命中句子": "",
            "命中章节": "",
            "确定性等级": "中",
            "判断原因": "未命中AI关键词，且行业明显偏非AI",
        }

    return {
        "与AI产业链是否有关": "否",
        "AI产业链环节": "",
        "命中关键词": "",
        "命中句子": "",
        "命中章节": "",
        "确定性等级": "中",
        "判断原因": "2025年年报未命中AI产业链关键词",
    }


# =========================
# 4. 主程序
# =========================

def main():
    print("开始读取A股名单...")
    stocks = read_stock_list(INPUT_FILE)

    if MAX_COMPANIES:
        stocks = stocks.head(MAX_COMPANIES)

    print(f"共读取公司数量：{len(stocks)}")
    print("开始批量搜索2025年年度报告并筛选AI产业链...")

    results = []

    for idx, row in tqdm(stocks.iterrows(), total=len(stocks)):
        code = row["股票代码"]
        name = row["股票简称"]

        report_info = cninfo_query_annual_report(code, name)

        pdf_path = PDF_DIR / f"{code}_{name}_{YEAR}年报.pdf"
        text_path = TEXT_DIR / f"{code}_{name}_{YEAR}年报.txt"

        text = ""

        if report_info:
            ok = download_pdf(report_info["年报链接"], pdf_path)
            if ok:
                text = extract_pdf_text(pdf_path, text_path)

        judgment = classify_company(row, report_info, text)

        item = {
            "股票代码": code,
            "股票简称": name,
            "所属行业": row.get("所属行业", ""),
            "报告年份": YEAR,
            "公告标题": report_info.get("公告标题", "") if report_info else "",
            "公告日期": report_info.get("公告日期", "") if report_info else "",
            "命中关键词": judgment["命中关键词"],
            "命中句子": judgment["命中句子"],
            "命中章节": judgment["命中章节"],
            "与AI产业链是否有关": judgment["与AI产业链是否有关"],
            "AI产业链环节": judgment["AI产业链环节"],
            "确定性等级": judgment["确定性等级"],
            "判断原因": judgment["判断原因"],
            "年报链接": report_info.get("年报链接", "") if report_info else "",
            "复核状态": "待复核" if judgment["与AI产业链是否有关"] in ["是", "不确定"] else "",
        }

        results.append(item)

        if len(results) % 20 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
            temp_df.to_excel(OUTPUT_EXCEL, index=False)

        time.sleep(SLEEP_SECONDS)

    result_df = pd.DataFrame(results)

    # 排序：是、不确定、否
    order_map = {"是": 1, "不确定": 2, "否": 3}
    result_df["_排序"] = result_df["与AI产业链是否有关"].map(order_map)
    result_df = result_df.sort_values(["_排序", "股票代码"]).drop(columns=["_排序"])

    result_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    result_df.to_excel(OUTPUT_EXCEL, index=False)

    print("完成！")
    print(f"Excel结果：{OUTPUT_EXCEL}")
    print(f"CSV结果：{OUTPUT_CSV}")
    print()
    print("统计结果：")
    print(result_df["与AI产业链是否有关"].value_counts(dropna=False))


if __name__ == "__main__":
    main()